# Hidden-size sweep — how do the four affordances scale with latent capacity?

**Direction:** `research/directions/hidden-size-sweep.md` · `[in-frame]` · sub-Q 1, 2, 3. **Model:** GRU only.

**The gap.** Every result in this repo — the identifiability numbers, the fiber residual, the whole §4 editability
negative — was measured at **one hidden size, `H = 256`**, chosen by default and never justified. This sweep varies only
that number and re-measures all four affordances.

**Reference points for reading the sweep.** The world's true state is **8 numbers** (2 objects × position and velocity
in 2D); the observation is **128 rays**. So `H = 8` sits exactly at the world's dimensionality, `H = 128` matches the
observation, and `H = 512` is 4× over-complete relative to the observation.

**Provenance.** All five models are trained by `scripts/train_gru.py` on `datasets/4_fixed_refl_inview` with an
identical recipe; the metric suite is computed once by `scripts/eval_controls.py` and only loaded and plotted here.

## Definitions

### Runs (copied from `CONTROL_RUNS.md`, per the repo's run-registry rule)

Every run: `datasets/4_fixed_refl_inview` (observation noise 0.2, position noise 0.04), 400 epochs, batch 256,
AdamW lr 1e-3, weight decay 1e-4, seed 0, 1 GRU layer, no dropout. **The only variable is `hidden_size`.**

| code | descriptive label (used in every figure) | hidden size | parameters | why this size |
|---|---|---|---|---|
| `H8` | **H=8** | 8 | 5,384 | the world's true state dimensionality — capacity-starved, forced toward canonicality |
| `H32` | **H=32** | 32 | 16,544 | well below observation resolution |
| `H128` | **H=128** | 128 | 148,352 | matches the observation resolution (128 rays) |
| `H256` | **H=256 (baseline)** | 256 | 460,672 | the repo default behind every earlier finding |
| `H512` | **H=512** | 512 | 1,707,648 | 4× over-complete relative to the observation |

### Metrics (formulas verbatim from `../METRICS_AND_EDITORS.md`)

| name | formula | units | better |
|---|---|---|---|
| next-step RMSE | `RMSE(pred_t, clean_obs[t+1])`, teacher-forced | obs intensity [0,1] | ↓ |
| free-run RMSE @ step s | warm up on `obs[0..9]`, then free-run; `RMSE(roll_s, clean_obs[10+s])` | obs intensity | ↓ |
| copy-previous-frame / noise floor / random frame | dataset baselines, `pim/eval/baselines.py` | obs intensity | reference lines |
| position / velocity R² | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, held-out 30%, frames where both objects are visible | — | ↑ |
| fiber residual | `‖h − g(pos,vel)‖ / ‖h‖`, `g` linear or MLP, held-out 30% | fraction of ‖h‖ | ↓ (0 = fully canonical) |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over the **differing rays**; per sample, then averaged | −1…+1 | ↑ |
| **Target RMSE** | `RMSE(edited₀, gt_edited)` over **target rays** | obs intensity | ↓ |
| **Ghost RMSE** | `RMSE(edited₀, gt_edited)` over **ghost rays** | obs intensity | ↓ |
| **Collateral RMSE** | `RMSE(edited₀, gt_edited)` over **collateral rays** | obs intensity | ↓ |
| **Edit-frame RMSE** | `RMSE(edited₀, gt_edited)` over **all rays** | obs intensity | ↓ |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the K-step rollout | obs intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ (**> 1 = the edit left the rollout FURTHER from the true post-edit world than doing nothing**) |

**The two ground-truth worlds.** Every §4 number is an error against **ground truth**, never against the unsteered
rollout. At the edit frame both worlds are rendered: **`gt_edited`** (= `clean_obs[ef]`, the teleport happened) and
**`gt_unedited`** (the counterfactual where it did not — the edited object continued from its `ef−1` position along its
own velocity, the other object at its true `ef` position).

**Ray zones (per sample, derived from those two renders, so occlusion needs no special-casing).** `target rays` = rays
the edited object occupies in `gt_edited`; `ghost rays` = rays it occupied pre-edit and now vacates; `collateral rays`
= the **other** object's rays (it must not move); `differing rays` = every ray where the two worlds differ — the
support of the Edit Index. **`ef` = 20**, **K = 15** rollout steps, **64** edit samples, **2000** test sequences for
probes.

> **How to read the Edit Index.** **+1** = the output *is* the edited world · **0** = equidistant from both (ambiguous,
> or garbage) · **−1** = the output *is* the unedited world. Unsteered lands near −1 by construction, so the scale is
> anchored on two ground truths rather than on a model-dependent reference. Crucially, an output far from *both* worlds
> — a scrambled or collapsed rollout — scores **≈ 0** rather than a spuriously good value, so the index cannot be gamed
> by destroying the output; the accompanying zone RMSEs then show *how* it was destroyed.
>
> Definitions: `../METRICS_AND_EDITORS.md` §4 · implementation: `scripts/editability_metrics.py` (imported, not
> re-derived). This set **replaces** the retired `reach % of swap` / `collateral % of swap` / `selectivity` /
> `ghost ratio` as of 2026-07-30 — their numbers are **not** comparable to these.

### Editors and references (the standard §4 suite)

**References (never editors):** **GT (sim)** — the simulator's time-evolving clean post-edit observations;
**Unsteered** — free-run from the un-edited warm-up state; **Oracle observation** — teacher-force the model on the true
post-edit observation; it is the 100% denominator for reach and a *soft* reference (one frame of teleport evidence only
partly updates the state).

| editor | mechanism |
|---|---|
| Readout injection | linear pseudoinverse — set the position probe's readout, preserving its null space |
| Global-PCA projection | POCS: alternate inject ↔ project onto the global 99%-variance PCA subspace of visited `h` |
| PCA geodesic | re-project onto a fresh **local** PCA tangent (k=256 neighbours) each step — the canonical structural editor |
| MLP-probe gradient | Adam on `h` through a frozen MLP position probe until it reads the target |
| Decoder gradient (**oracle**) | Adam on `h` to match the true edit-frame observation through the decoder. Not an edit interface — it is the **bracket**: it shows whether a state that renders the target exists and rolls out at all |

> **±1 alignment.** Warm-up teacher-forces `obs[0..ef−1]`, so a rollout's **step 0 decodes sim frame `ef`**
> (`ROLL[:,0] ↔ clean_obs[ef]`). The oracle observation is fed `obs[ef]` (repo convention) and therefore **leads by one
> frame**; recorded in the JSON as `swap_frame_lead`.

In [ ]:
# [1] Setup: load the pre-computed metric suite (scripts/eval_controls.py) for all five hidden sizes.
import os, sys, json
sys.path.insert(0, "../../../..")
import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown
from pim.figures.theme import style_ax

OUT = "/tmp/hidden_size_sweep"; os.makedirs(OUT, exist_ok=True)
EVAL = "../../../../runs/controls/eval"

RUNS  = ["H8", "H32", "H128", "H256", "H512"]
LABEL = {"H8": "H=8", "H32": "H=32", "H128": "H=128", "H256": "H=256 (baseline)", "H512": "H=512"}
COLOR = {"H8": "#0072B2", "H32": "#009E73", "H128": "#E69F00", "H256": "#D55E00", "H512": "#CC79A7"}
EDITORS = ["Readout injection", "Global-PCA projection", "PCA geodesic",
           "MLP-probe gradient", "Decoder gradient"]
STRUCTURAL = EDITORS[:4]          # probe-directed write mechanisms
ORACLE = "Decoder gradient"       # the bracket, not an edit interface

RES = {r: json.load(open(f"{EVAL}/{r}.json")) for r in RUNS}
NPZ = {r: np.load(f"{EVAL}/{r}_rollouts.npz") for r in RUNS}
HS  = np.array([RES[r]["hidden_size"] for r in RUNS])
K   = len(RES["H256"]["editability"]["PCA geodesic"]["step_rmse_to_gt"])
ef  = int(NPZ["H256"]["edit_frame"][0])

print(f"loaded {len(RUNS)} runs | hidden sizes {list(HS)} | K={K} rollout steps | edit frame ef={ef}")
for r in RUNS:
    d = RES[r]
    print(f"  {LABEL[r]:<18s} {d['n_params']:>9,} params | val_loss {d['val_loss']:.5f} | "
          f"next-step RMSE {d['nextstep_rmse_vs_clean']:.4f}")
bl = RES["H256"]["baselines"]   # identical for every run — all five share dataset 4
print(f"\ndataset baselines (shared by all five runs): copy-previous-frame {bl['identity_rmse']:.4f} | "
      f"observation noise floor {bl['noise_floor_rmse']:.4f} | random frame {bl['random_rmse']:.4f}")

---
## §1 — Predictive quality

Does capacity buy prediction, and where does it saturate? All five runs share one dataset, so a single set of dashed
baselines applies to every curve.

In [ ]:
# [2] Fig 1 — predictive quality: (a) validation curves, (b) free-run RMSE vs step, (c) next-step RMSE vs capacity.
curves = {}
for r in RUNS:
    hist = [json.loads(l) for l in open(f"../../../../runs/controls/{r}/metrics.jsonl")]
    curves[r] = ([h["epoch"] for h in hist], [h["train_loss"] for h in hist], [h["val_loss"] for h in hist])

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.3))
for r in RUNS:
    e, tr, va = curves[r]
    ax[0].plot(e, va, color=COLOR[r], lw=1.5, label=LABEL[r])
    ax[0].plot(e, tr, color=COLOR[r], lw=0.8, ls=":", alpha=0.7)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("teacher-forced MSE vs next noisy frame"); ax[0].set_yscale("log")
ax[0].set_title("(a) training curves\n(solid = validation, dotted = train)", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3); style_ax(ax[0])

s = np.arange(len(RES["H256"]["freerun_rmse_by_step"]))
for r in RUNS:
    ax[1].plot(s, RES[r]["freerun_rmse_by_step"], "-o", ms=3, color=COLOR[r], label=LABEL[r])
for v, c, lab in [(bl["identity_rmse"], "#8B4513", "copy previous frame"),
                  (bl["noise_floor_rmse"], "#333333", "observation noise floor"),
                  (bl["random_rmse"], "0.5", "random frame")]:
    ax[1].axhline(v, ls="--", lw=1.1, color=c, label=f"{lab} ({v:.3f})")
ax[1].set_xlabel("free-run step (0 = first unobserved frame)"); ax[1].set_ylabel("RMSE vs clean observation [0,1]")
ax[1].set_title("(b) how fast the free run decays", fontsize=10)
ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3); style_ax(ax[1])

ax[2].plot(HS, [RES[r]["nextstep_rmse_vs_clean"] for r in RUNS], "-o", color="#0072B2", label="next-step RMSE")
ax[2].plot(HS, [RES[r]["freerun_rmse_by_step"][-1] for r in RUNS], "-s", color="#CC79A7",
           label=f"free-run RMSE at step {len(s)-1}")
ax[2].axhline(bl["noise_floor_rmse"], ls="--", lw=1.1, color="#333333", label="observation noise floor")
ax[2].set_xscale("log", base=2); ax[2].set_xticks(HS); ax[2].set_xticklabels(HS)
ax[2].set_xlabel("hidden size H"); ax[2].set_ylabel("RMSE vs clean observation [0,1]")
ax[2].set_title("(c) prediction versus capacity", fontsize=10)
ax[2].legend(fontsize=7.5); ax[2].grid(alpha=0.3); style_ax(ax[2])
fig.suptitle("Fig 1 — predictive quality across hidden size (all five runs share dataset 4, so baselines are shared)",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_predictive.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §2 / §3 — Recoverability and canonicality versus capacity

Capacity is the x-axis here, not a series: the question is a *trend*, so one line per metric reads better than five
lines per panel. The **linear** probe is the informative axis — the MLP probe saturates on this task.

In [ ]:
# [3] Fig 2 + Table 1 — position/velocity R² and fiber residual as a function of hidden size.
plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.3))
for key, lab, col, mk in [("pos_r2_linear", "position R² (linear probe)", "#0072B2", "o"),
                          ("pos_r2_mlp",    "position R² (MLP probe)",    "#0072B2", "s"),
                          ("vel_r2_linear", "velocity R² (linear probe)", "#D55E00", "o"),
                          ("vel_r2_mlp",    "velocity R² (MLP probe)",    "#D55E00", "s")]:
    ax[0].plot(HS, [RES[r][key] for r in RUNS], marker=mk,
               ls=("-" if "linear" in key else "--"), color=col, ms=5, label=lab)
ax[0].set_ylabel("R² (higher is better)"); ax[0].set_title("(a) recoverability of the physical state", fontsize=10)
ax[0].axhline(0, color="0.3", lw=0.8)

for key, lab, col, ls in [("fiber_resid_linear", "fiber residual (linear)", "#009E73", "-"),
                          ("fiber_resid_mlp",    "fiber residual (MLP)",    "#009E73", "--")]:
    ax[1].plot(HS, [RES[r][key] for r in RUNS], ls, marker="o", color=col, ms=5, label=lab)
ax[1].set_ylabel("residual as a fraction of ‖h‖ (lower is better)")
ax[1].set_title("(b) canonicality: how much of h is NOT a\nfunction of (position, velocity)?", fontsize=10)
ax[1].set_ylim(0, None)

for key, lab, col in [("vel_r2_linear", "velocity R² (linear), all frames", "#D55E00"),
                      ("late_vel_r2_linear", "velocity R² (linear), late frames t ≥ 15", "#56B4E9")]:
    ax[2].plot(HS, [RES[r][key] for r in RUNS], "-o", color=col, ms=5, label=lab)
ax[2].set_ylabel("R² (higher is better)")
ax[2].set_title("(c) velocity is the hard coordinate:\nearly frames vs the filter-converged regime", fontsize=10)

for a in ax:
    a.set_xscale("log", base=2); a.set_xticks(HS); a.set_xticklabels(HS)
    a.set_xlabel("hidden size H"); a.legend(fontsize=7.5); a.grid(alpha=0.3); style_ax(a)
fig.suptitle("Fig 2 — what the hidden state encodes, as a function of capacity", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recovery.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| model | params | next-step RMSE ↓ | position R² lin / MLP ↑ | velocity R² lin / MLP ↑ | fiber residual lin / MLP ↓ |",
        "|---|---|---|---|---|---|"]
for r in RUNS:
    d = RES[r]
    rows.append(f"| {LABEL[r]} | {d['n_params']:,} | {d['nextstep_rmse_vs_clean']:.4f} | "
                f"{d['pos_r2_linear']:.3f} / {d['pos_r2_mlp']:.3f} | "
                f"{d['vel_r2_linear']:.3f} / {d['vel_r2_mlp']:.3f} | "
                f"{d['fiber_resid_linear']:.3f} / {d['fiber_resid_mlp']:.3f} |")
display(Markdown("**Table 1 — predictive quality, recoverability and canonicality across capacity**\n\n" + "\n".join(rows)))

---
## §4 — Editability versus capacity

The decisive axis is **ghost ratio**: 1.0 means the object never left its old location, whatever else the edit did to
the observation. The **decoder-gradient oracle** is plotted alongside the probe-directed editors as the bracket — it is
not an edit interface, it is the check that a state rendering the target *exists and rolls out* at this capacity. If the
structural editors sit at ghost ≈ 1.0 while the oracle does not, the failure is the reachability of the edit map rather
than the model.

In [ ]:
# [4] Fig 3 — editability vs hidden size: the Edit Index headline, then the zone decomposition that explains it.
plt.style.use("default")
EC = {"Readout injection": "#0072B2", "Global-PCA projection": "#E69F00", "PCA geodesic": "#CC79A7",
      "MLP-probe gradient": "#56B4E9", "Decoder gradient": "#009E73"}
panels = [("edit_index",      "Edit Index  (+1 edited world … −1 unedited world)",
           "(a) HEADLINE — did the edit land?"),
          ("target_rmse",     "Target RMSE vs ground truth",
           "(b) did the object appear at the target?"),
          ("ghost_rmse",      "Ghost RMSE vs ground truth",
           "(c) did it leave its old location?"),
          ("collateral_rmse", "Collateral RMSE vs ground truth",
           "(d) was the OTHER object left alone?"),
          ("gt_traj_rmse",    "GT-traj RMSE (mean over the rollout)",
           "(e) did the edit HOLD over the rollout?")]
fig, ax = plt.subplots(1, 5, figsize=(26, 4.4))
for i, (key, ylab, ttl) in enumerate(panels):
    for ed in EDITORS:
        lw = 2.4 if ed == ORACLE else 1.6
        ax[i].plot(HS, [RES[r]["editability"][ed][key] for r in RUNS], "-o", ms=4.5, lw=lw,
                   color=EC[ed], label=ed + (" (ORACLE)" if ed == ORACLE else ""))
    ax[i].plot(HS, [RES[r]["editability"]["Unsteered"][key] for r in RUNS], ":", color="0.55", lw=1.6,
               label="unsteered (no edit)")
    ax[i].plot(HS, [RES[r]["editability"]["Oracle observation"][key] for r in RUNS], "--", color="0.2", lw=1.6,
               label="oracle observation (reference)")
    ax[i].set_xscale("log", base=2); ax[i].set_xticks(HS); ax[i].set_xticklabels(HS)
    ax[i].set_xlabel("hidden size H"); ax[i].set_ylabel(ylab); ax[i].set_title(ttl, fontsize=10)
    ax[i].grid(alpha=0.3); style_ax(ax[i])
for y, lab in [(1.0, "edited world"), (0.0, "equidistant / garbage"), (-1.0, "unedited world")]:
    ax[0].axhline(y, color="0.4", ls=":", lw=1.0)
    ax[0].annotate(lab, xy=(HS[-1], y), fontsize=7, color="0.35", ha="right", va="bottom")
ax[0].set_ylim(-1.05, 1.05)
# observation intensity is bounded in [0,1]: a zone RMSE above 1 means the edit blew the scan out of range
for i in (1, 2, 3):
    ax[i].axhline(1.0, color="#D55E00", ls=":", lw=1.2)
    ax[i].annotate("RMSE > 1: observation destroyed\n(intensity is bounded in [0,1])",
                   xy=(HS[0], 1.02), fontsize=6.5, color="#D55E00", va="bottom")
h, l = ax[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=7, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 0.99))
fig.suptitle("Fig 3 — editability across capacity: probe-directed editors versus the decoder-gradient oracle bracket",
             y=1.07, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93]); fig.savefig(f"{OUT}/fig3_editability.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

# Fig 3b — the Edit Index over the ROLLOUT: a step-0 win is not the same as an edit that holds.
fig, axes = plt.subplots(1, len(RUNS), figsize=(4.0*len(RUNS), 3.9), sharey=True)
sx = np.arange(K)
for a, r in zip(axes, RUNS):
    e = RES[r]["editability"]
    for ed in EDITORS:
        a.plot(sx, e[ed]["edit_index_by_step"], lw=2.4 if ed == ORACLE else 1.5, color=EC[ed],
               label=ed + (" (ORACLE)" if ed == ORACLE else ""))
    a.plot(sx, e["Unsteered"]["edit_index_by_step"], ":", color="0.55", lw=1.6, label="unsteered (no edit)")
    a.plot(sx, e["Oracle observation"]["edit_index_by_step"], "--", color="0.2", lw=1.6,
           label="oracle observation (reference)")
    a.axhline(0, color="0.4", ls=":", lw=1.0); a.set_ylim(-1.05, 1.05)
    a.set_title(LABEL[r], fontsize=10); a.set_xlabel("rollout step s (0 = frame ef)")
    a.grid(alpha=0.3); style_ax(a)
axes[0].set_ylabel("Edit Index")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=7, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Fig 3b — does the edit HOLD? Edit Index at every rollout step "
             "(+1 = the edited world, −1 = the unedited world, 0 = neither)", y=1.12, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.90]); fig.savefig(f"{OUT}/fig3b_edit_index_rollout.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [5] Fig 4 — per-step RMSE against the moving post-edit ground truth: one panel per editor, one line per H.
plt.style.use("default")
panels = ["Unsteered", "Oracle observation"] + EDITORS
fig, axes = plt.subplots(2, 4, figsize=(18, 7.6), sharex=True, sharey=True)
s = np.arange(K)
for a, ed in zip(axes.ravel(), panels):
    for r in RUNS:
        a.plot(s, RES[r]["editability"][ed]["step_rmse_to_gt"], color=COLOR[r], lw=1.6, label=LABEL[r])
    ttl = ed + (" (ORACLE — the bracket)" if ed == ORACLE else
                (" (reference)" if ed in ("Unsteered", "Oracle observation") else ""))
    a.set_title(ttl, fontsize=9.5); a.grid(alpha=0.3); style_ax(a)
for a in axes[-1]: a.set_xlabel("rollout step s (0 = sim frame ef)")
for a in axes[:, 0]: a.set_ylabel("RMSE vs clean_obs[ef+s]")
axes.ravel()[-1].axis("off")
h, l = axes[0][0].get_legend_handles_labels()
fig.legend(h, l, loc="lower right", ncol=1, fontsize=10, frameon=False, bbox_to_anchor=(0.93, 0.13))
fig.suptitle("Fig 4 — does the edit land and hold? RMSE against the time-evolving true post-edit observation",
             y=1.0, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_steprmse.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [6] Canonical observation-waterfall helper (CLAUDE.md fixed spec): gray on dark, 6 noisy context frames above a
#     dashed edit line, the shared TRUE post-edit row, then each column's OWN free-run; green target / red-dash ghost.
N_CTX = 6
DARK, TXT, TICK, EDIT_C, ROW_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
TARGET_C, GHOST_C = "#00E676", "#FF5252"

def waterfall_grid(col_titles, rows, suptitle, fname):
    """rows[i] = dict(label, ctx (N_CTX,R), bodies list[(K,R)], tgt_cx, ghost_cx).

    Each column shows its OWN free-run from step 0 (which decodes sim frame `ef`) — there is
    deliberately NO shared teacher-forced `ef` row: only the Oracle observation reference ever
    sees that frame, and painting it into every column would hide the exact frame §4 scores.
    Each row is self-contained (own context frames, locators and GT column), which is what lets
    rows come from different models/datasets as well as from different samples."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(rows), ncol, figsize=(2.9*ncol, 3.3*len(rows)),
                             squeeze=False, facecolor=DARK)
    for r, row in enumerate(rows):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([row["ctx"], row["bodies"][c]], 0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX-0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(row["tgt_cx"]):   ax.axvline(row["tgt_cx"],   color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(row["ghost_cx"]): ax.axvline(row["ghost_cx"], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(row["label"], fontsize=7.5, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX+7, N_CTX+14])
                ax.set_yticklabels([ef-N_CTX, ef, ef+7, ef+14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0],[0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)

# the three edits with the largest teleport that also vacate enough rays to make a ghost visible
z = NPZ["H256"]
SAMPLES = list(np.argsort(z["teleport"] * (z["n_ghost_rays"] >= 3))[::-1][:3])
print("waterfall samples (largest teleports):", SAMPLES,
      "| teleport distances:", np.round(z["teleport"][SAMPLES], 2))

In [ ]:
# [7] Fig 5 + Fig 6 — waterfalls: the canonical structural editor, then the oracle bracket, across capacity.
def rows_for(editor):
    out = []
    for smp in SAMPLES:
        z = NPZ["H256"]
        out.append(dict(
            label=f"sample {smp}\n(teleport {z['teleport'][smp]:.1f})\nsim frame",
            ctx=z["ctx"][smp],
            tgt_cx=z["tgt_cx"][smp], ghost_cx=z["ghost_cx"][smp],
            bodies=[NPZ["H256"]["gt_roll"][smp], NPZ["H256"]["roll_Unsteered"][smp]]
                   + [NPZ[r][f"roll_{editor}"][smp] for r in RUNS]))
    return out

cols = ["GT (sim)", "unsteered\n(no edit)"] + [LABEL[r] for r in RUNS]
waterfall_grid(cols, rows_for("PCA geodesic"),
               "Fig 5 — PCA geodesic (the canonical structural editor) at every hidden size: "
               "does the object move and does the old copy clear?", "fig5_waterfall_geodesic.png")
waterfall_grid(cols, rows_for("Decoder gradient"),
               "Fig 6 — decoder-gradient ORACLE at every hidden size (the bracket: a state that renders the target "
               "exists and rolls out)", "fig6_waterfall_oracle.png")

---
## §5 — Summary

Read three things off the numbers below: where prediction saturates, whether the latent becomes *more* linearly
readable and *more* canonical as capacity shrinks, and whether **any** capacity makes the latent grabbable.

In [ ]:
# [8] Table 2 + data-driven verdict, on the canonical §4 set (Edit Index + zone RMSEs + fidelity ratio).
rows = ["| model | editor | Edit Index ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio |",
        "|---|---|---|---|---|---|---|---|"]
BEST, VALID = {}, {}
for r in RUNS:
    e = RES[r]["editability"]
    # a structural editor only counts as having made an edit if it did not DEGRADE the rollout
    # (fidelity ratio <= 1); otherwise a move toward index 0 is destruction, not relocation.
    ok = [ed for ed in STRUCTURAL if e[ed]["fidelity_ratio"] <= 1.0]
    BEST[r] = max(ok or STRUCTURAL, key=lambda ed: e[ed]["edit_index"])
    VALID[r] = bool(ok)
    for ed in ["Unsteered", "Oracle observation"] + STRUCTURAL + [ORACLE]:
        c = e[ed]
        note = " *(destroys the observation)*" if max(c["target_rmse"], c["ghost_rmse"]) > 1.0 else ""
        rows.append(f"| {LABEL[r]} | {ed}{note} | **{c['edit_index']:+.2f}** | {c['target_rmse']:.3f} | "
                    f"{c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | "
                    f"{c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 2 — editability across capacity.** Edit Index: +1 = the output is the world where the edit "
                 "happened, −1 = the world where it did not, 0 = equidistant from both (ambiguous or garbage). "
                 "Observation intensity is bounded in [0,1], so a zone RMSE above 1 means the edit pushed the scan "
                 "out of range entirely.\n\n" + "\n".join(rows)))

print("================ VERDICT (computed, not asserted) ================")
ns = [RES[r]["nextstep_rmse_vs_clean"] for r in RUNS]
best_ns = min(ns)
sat = next((HS[i] for i in range(len(HS)) if ns[i] <= best_ns * 1.05), HS[-1])
print(f"1. PREDICTION SATURATES BY H={sat}: next-step RMSE is within 5% of the sweep best ({best_ns:.4f}) from there on.")
print("   next-step RMSE by H: " + ", ".join(f"H={h} {v:.4f}" for h, v in zip(HS, ns)))

pl = [RES[r]["pos_r2_linear"] for r in RUNS]; vl = [RES[r]["vel_r2_linear"] for r in RUNS]
fl = [RES[r]["fiber_resid_linear"] for r in RUNS]; fm = [RES[r]["fiber_resid_mlp"] for r in RUNS]
print(f"2. READABILITY RISES MONOTONICALLY WITH CAPACITY: linear position R² {pl[0]:.3f} (H=8) -> {pl[-1]:.3f} (H=512), "
      f"linear velocity R² {vl[0]:.3f} -> {vl[-1]:.3f}.")
print(f"   A capacity-starved latent is therefore NOT a more linearly readable one -- it simply fails to represent the "
      f"state. Canonicality moves the other way: fiber residual linear {fl[0]:.3f} -> {fl[-1]:.3f}, "
      f"MLP {fm[0]:.3f} -> {fm[-1]:.3f}, so large-H states carry proportionally more content that is not a function "
      f"of (position, velocity).")

print("3. EDITABILITY -- Edit Index, best probe-directed editor vs the oracle, by capacity:")
for r, h in zip(RUNS, HS):
    e = RES[r]["editability"]
    b = BEST[r]; cb, cu, co = e[b], e["Unsteered"], e[ORACLE]
    wrecked = max(cb["target_rmse"], cb["ghost_rmse"]) > 1.0
    print(f"     H={h:<4d} unsteered {cu['edit_index']:+.2f} | best structural {cb['edit_index']:+.2f} ({b})"
          f"{'  <- NONE pass the fidelity guard; this one degrades the rollout' if wrecked else ''} | oracle {co['edit_index']:+.2f}")
gap = [RES[r]["editability"][BEST[r]]["edit_index"] - RES[r]["editability"]["Unsteered"]["edit_index"] for r in RUNS]
print(f"   Best probe-directed movement away from 'did nothing', by H: "
      + ", ".join(f"H={h} {g:+.2f}" for h, g in zip(HS, gap)))
oi = [RES[r]["editability"][ORACLE]["edit_index"] for r in RUNS]
print(f"   Oracle Edit Index by H: " + ", ".join(f"H={h} {v:+.2f}" for h, v in zip(HS, oi))
      + "  (rises with capacity -- a bigger latent makes the target state more precisely reachable by decoder optimisation)")
print("   => " + ("NO capacity in this sweep makes the latent grabbable: every probe-directed editor stays near the "
                  "unsteered end of the index (or lands at ~0 by destroying the observation), while the "
                  "decoder-gradient oracle reaches the edited world on the same model and decoder at every H. "
                  "The failure is the reachability of the edit map, not capacity."
                  if max(gap) < 0.6 and min(oi) > 0.5 else
                  "the capacity axis DOES move grabbability -- inspect Table 2, Fig 3a and Fig 5 before concluding."))
print("\nPNGs:", sorted(os.listdir(OUT)))